### Imports

In [ ]:
from electron_state import ElectronState, etensor
from qutip_objects import kick, free_electron
from qutip import *
import matplotlib.pyplot as plt
from plot_utils import plot_2d_field
from electron_wigner import ewigner
from physics_utils import PhysicalQuantities
import numpy as np

# Global definitions

## Computation constants

These numbers are used throughout the whole notebook.

For high n values (>80), the computations become significantly slower.

The g0 parameter can be a complex number.

In [ ]:
# Computed electron energy levels
n_electron = 51
# Computed photon energy levels
n_photon = 50

# Coupling constant
g0 = 1.0

# Object to define all the relevant physical quantities
phq = PhysicalQuantities(lambda_p=515, # in nm
                         high_tension=200, # in kV
                         zlp_fwhm=0.7, # in eV
                         momentum_resolution=40, # number of points per energy levels
                         unit_system='atomic')


# S-matrix

The electron-light interaction is represented by the S-matrix. This operator is used for every example of this notebook.

In [ ]:
a = tensor(destroy(n_photon), qeye(n_electron))
b = tensor(qeye(n_photon),kick(n_electron,1))
ps = g0*b*a.dag()  - np.conj(g0)*b.dag()*a
s = ps.expm()

### Electron state

The electron before interaction is in a state with energy $E_0$ (without taking into account the energy dispersion). The initial ket is then $\ket{E_0}$ an energy loss of $\hbar \omega$ is $\ket{E_{-1}}$ and an energy gain of $\hbar \omega$ is $\ket{E_1}$. This Hilbert space is very different from the Hilbert space of the photon. We use the `ElectronState` of pyiels to perform the calculation in the correct space for the electron.

The same initial electron state is used throughout the whole notebook.

In [ ]:
electron = ElectronState(free_electron(n_electron,0))

# Fig. 1. a. and b.

These figures present the interaction between a coherent state of light and a free electron.

## Coherent state

This group of cells computes the wigner functions of a coherent state of photons in interaction with a free electron.

### Photon state

In [ ]:
# modulus of the coherent state parameter
alpha_mod=np.sqrt(5)
# Phase of the coherent state
theta=np.pi/4

# Complex coherent state parameter
alpha=alpha_mod*np.exp(1j*theta)

# Displace operator
disp = displace(n_photon, alpha)

# Coherent photon state
coherent_photon=disp*basis(n_photon,0)

## Fig. 1. a. Photon coherent state wigner function

In [ ]:
plot_wigner(coherent_photon,np.linspace(-5,5,200),np.linspace(-5,5,200))
plt.axhline(0,color = 'grey', alpha = 0.2, linestyle = '--')
plt.axvline(0,color = 'grey', alpha = 0.2,linestyle = '--')


## Initial state preparation

### Tensor state electron photon

In pyiels you can create electron-photon tensor states.

In [ ]:
init_epc = etensor(coherent_photon,electron)

### Electron-photon density matrix

In [ ]:
dm_epc = init_epc*init_epc.dag()

## Electron-photon interaction

We apply the s-matrix operator.

In [ ]:
final_epc = s*dm_epc*s.dag()

### Trace on photon after interaction

To retrieve the electron state after interaction, we perform the partial trace on the photon.

In [ ]:
final_ec = final_epc.ptrace([1])

## Fig 1. b. Electron wigner function after interaction with a coherent photon state

In [ ]:
w, ps,z = ewigner(final_ec,phq=phq,energy_levels=True)

plot_2d_field(w,z,ps,
              ylim = (-12,12),
              xlim = (0,2*np.pi),
              normalize = True,
              use_pi_ticks = True,
              use_enl_ticks = True,
              unit_system='',
              grid=True,
              projection = False)

# Fig. 1. c. and d.

These figures present the interaction between a vacuum light state and a free electron

## Vacuum photon state

In [ ]:
# vacuum state
vacuum_photon = basis(n_photon,0)

## Fig. 1. c. Photon vacuum state wigner function

In [ ]:
plot_wigner(vacuum_photon,np.linspace(-5,5,200),np.linspace(-5,5,200))
plt.axhline(0,color = 'grey', alpha = 0.2, linestyle = '--')
plt.axvline(0,color = 'grey', alpha = 0.2,linestyle = '--')

### Light-electron interaction

In [ ]:
init_epv = etensor(vacuum_photon,electron)
dm_epv = init_epv*init_epv.dag()
final_epv = s*dm_epv*s.dag()
final_ev = final_epv.ptrace([1])

## Fig 1. d. Electron wigner function after interaction with a vacuum photon state

In [ ]:
w, ps,z = ewigner(final_ev,phq=phq,energy_levels=True)

plot_2d_field(w,z,ps,
              ylim = (-12,12),
              xlim = (0,2*np.pi),
              normalize = True,
              use_pi_ticks = True,
              use_enl_ticks = True,
              unit_system='',
              grid=True,
              projection = False)

# Fig. 1. e. and f.

These figures present the interaction between a squeezed vacuum light state and a free electron

## Squeezed photon state

In [ ]:
z = 1.0*np.exp(-1j*np.pi/3) # Squeezing parameter (complex number)

sq = squeeze(n_photon, z) # Squeezing operator
vac = basis(n_photon, 0) # Vacuum state

# Squeezed vacuum state
squeezed_photon = sq * vac

## Fig. 1. e. Photon squeezed vacuum state wigner function

In [ ]:
plot_wigner(squeezed_photon,np.linspace(-5,5,200),np.linspace(-5,5,200))
plt.axhline(0,color = 'grey', alpha = 0.2, linestyle = '--')
plt.axvline(0,color = 'grey', alpha = 0.2,linestyle = '--')

### Light-electron interaction

In [ ]:
init_eps = etensor(squeezed_photon,electron)
dm_eps = init_eps*init_eps.dag()
final_eps = s*dm_eps*s.dag()
final_es = final_eps.ptrace([1])

## Fig 1. f. Electron wigner function after interaction with a vacuum photon state

In [ ]:
w, ps,z = ewigner(final_es,phq=phq,energy_levels=True)

plot_2d_field(w,z,ps,
              ylim = (-12,12),
              xlim = (0,2*np.pi),
              normalize = True,
              use_pi_ticks = True,
              use_enl_ticks = True,
              unit_system='',
              grid=True,
              projection = False)